In [1]:
!pip install -q torch torchvision transformers pillow pandas


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from typing import Optional

import pandas as pd
import torch
from IPython.display import display
from PIL import Image, UnidentifiedImageError
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Prompt untuk kelas "makanan asli": ciri khas foto amatir/natural
REAL_PROMPTS = [
    "a real photo of food taken with a smartphone camera",
    "a candid photograph of a home-cooked meal on a table",
    "an amateur food photo with natural lighting and slight imperfections",
    "a photo of real food with natural crumbs, grease and uneven texture",
    "a restaurant dish photographed by a customer",
]

# Prompt untuk kelas "buatan AI": ciri visual yang sering muncul di gambar AI
AI_PROMPTS = [
    "an AI-generated image of food",
    "a digitally rendered, unrealistically perfect dish",
    "a food image with overly smooth plastic-like texture",
    "a hyper-detailed surreal food picture made by midjourney",
    "a computer-generated meal with glossy waxy surfaces and distorted shapes",
]

c:\Users\SKYL15H\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.10.2). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [3]:
class FoodAuthenticityDetector:
    """Membungkus model CLIP untuk klasifikasi zero-shot: makanan asli vs buatan AI."""

    def __init__(self, model_name: str = MODEL_NAME, device: Optional[str] = None):
        # Pakai GPU kalau tersedia, kalau tidak pakai CPU
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[info] Memuat model {model_name} di {self.device} ...")
        try:
            self.model = CLIPModel.from_pretrained(model_name).to(self.device).eval()
            self.processor = CLIPProcessor.from_pretrained(model_name)
        except Exception as e:  # biasanya masalah internet saat download pertama
            raise RuntimeError(f"Gagal memuat model: {e}")

        # Gabungkan semua prompt; ingat batas indeks antara kelas asli dan AI
        self.prompts = REAL_PROMPTS + AI_PROMPTS
        self.n_real = len(REAL_PROMPTS)
        print("[info] Model siap dipakai ✅")

    @staticmethod
    def load_image(path) -> Optional[Image.Image]:
        """Buka gambar dengan aman. Mengembalikan None kalau gagal."""
        try:
            # convert("RGB") penting: PNG transparan / grayscale bisa bikin CLIP error
            return Image.open(path).convert("RGB")
        except (FileNotFoundError, UnidentifiedImageError, OSError) as e:
            print(f"[warn] Lewati '{Path(path).name}': gagal dimuat ({e})")
            return None

    @torch.no_grad()  # kita tidak training, jadi matikan gradien (hemat memori)
    def predict(self, image: Image.Image) -> dict:
        # Ubah gambar + semua teks menjadi tensor
        inputs = self.processor(
            text=self.prompts, images=image, return_tensors="pt", padding=True
        ).to(self.device)

        # logits_per_image: skor kemiripan gambar vs tiap kalimat, bentuk (1, jumlah_prompt)
        logits = self.model(**inputs).logits_per_image[0]

        # Rata-ratakan skor per kelas, lalu softmax antar 2 kelas
        real_score = logits[: self.n_real].mean()
        ai_score = logits[self.n_real :].mean()
        probs = torch.softmax(torch.stack([real_score, ai_score]), dim=0)

        # Probabilitas tiap prompt (berguna untuk debugging)
        per_prompt = torch.softmax(logits, dim=0).tolist()

        return {
            "real": probs[0].item(),
            "ai": probs[1].item(),
            "per_prompt": dict(zip(self.prompts, per_prompt)),
        }


def verdict(result: dict, threshold: float = 0.60) -> str:
    """Ubah probabilitas menjadi label. Di bawah threshold = 'tidak yakin'."""
    top = max(result["real"], result["ai"])
    if top < threshold:
        return "🤔 TIDAK YAKIN"
    return "📸 MAKANAN ASLI" if result["real"] > result["ai"] else "🤖 BUATAN AI"


detector = FoodAuthenticityDetector()

[info] Memuat model openai/clip-vit-base-patch32 di cuda ...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[info] Model siap dipakai ✅


In [4]:
# --- (Opsional, khusus Google Colab) unggah foto lewat browser ---
# from google.colab import files
# uploaded = files.upload()
# IMAGE_PATH = list(uploaded.keys())[0]

IMAGE_PATH = "foto_makanan.jpg"   # <-- ganti dengan path fotomu
THRESHOLD = 0.70                  # batas keyakinan minimum
SHOW_DETAIL = True                # tampilkan skor tiap prompt?

image = detector.load_image(IMAGE_PATH)
if image is not None:
    # Tampilkan thumbnail (copy dulu supaya gambar asli tidak berubah)
    preview = image.copy()
    preview.thumbnail((400, 400))
    display(preview)

    result = detector.predict(image)
    print(f"Hasil     : {verdict(result, THRESHOLD)}")
    print(f"Asli      : {result['real']:.1%}")
    print(f"Buatan AI : {result['ai']:.1%}")

    if SHOW_DETAIL:
        print("\nDetail per prompt:")
        for prompt, p in result["per_prompt"].items():
            print(f"  {p:6.1%}  {prompt}")

[warn] Lewati 'foto_makanan.jpg': gagal dimuat ([Errno 2] No such file or directory: 'foto_makanan.jpg')


In [6]:
FOLDER_PATH = "folder_foto"   # <-- ganti dengan folder berisi foto-fotomu
THRESHOLD = 0.60

folder = Path(FOLDER_PATH)
if not folder.is_dir():
    print(f"[error] Folder '{FOLDER_PATH}' tidak ditemukan.")
else:
    rows = []
    for img_path in sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS):
        img = detector.load_image(img_path)
        if img is None:
            continue
        try:
            res = detector.predict(img)
        except Exception as e:
            print(f"[warn] Gagal memproses '{img_path.name}': {e}")
            continue
        rows.append({
            "file": img_path.name,
            "hasil": verdict(res, THRESHOLD),
            "prob_asli": round(res["real"], 3),
            "prob_ai": round(res["ai"], 3),
        })

    if rows:
        df = pd.DataFrame(rows)
        display(df)
        # df.to_csv("hasil_deteksi.csv", index=False)  # hapus '#' untuk simpan ke CSV
    else:
        print("[info] Tidak ada gambar yang berhasil diproses.")

,file,hasil,prob_asli,prob_ai
0,AI-1.jpg,📸 MAKANAN ASLI,0.740,0.260
1,AI-10.jpg,📸 MAKANAN ASLI,0.660,0.340
2,AI-11.jpg,📸 MAKANAN ASLI,0.604,0.396
3,AI-12.jpg,🤖 BUATAN AI,0.378,0.622
4,AI-13.jpg,📸 MAKANAN ASLI,0.679,0.321
5,AI-14.jpg,📸 MAKANAN ASLI,0.697,0.303
6,AI-15.jpg,🤖 BUATAN AI,0.198,0.802
7,AI-16.jpg,🤔 TIDAK YAKIN,0.408,0.592
8,AI-17.jpg,🤖 BUATAN AI,0.208,0.792
9,AI-18.jpg,📸 MAKANAN ASLI,0.824,0.176
